In [1]:
"""qPCR copies/mL vs. sequencing RPKMF, PER-VIRUS -- 10 plots total (5 viruses
x {alldata, double-positive}). 

Two variants per virus:
  - alldata: ALL samples 
    quad-marginal jointplot (main scatter + KDE violin marginals on all four
    sides) -- symlog x/y so exact zeros sit at the axis origin. Top/right
    marginals show the log-distribution of that axis's own positive values;
    left/bottom show the other assay's value among that assay's negative
    (exact zero) calls.
    
  - doublepositive: qpcr_detected == "detected" AND sequencing_detected == "detected" 
    only -- log-log scatter + top/right marginals + log-log OLS fit line.

Outputs (folder-per-virus, matching the established FluA/FluB/RsvA/RsvB/
SARS-COV-2 convention):
  fig3b_copiesperml_vs_rpkmf_{slug}_alldata_data3.png
  fig3b_copiesperml_vs_rpkmf_{slug}_doublepositive_data3.png
"""

#Import required packaged
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpecFromSubplotSpec
from scipy.stats import gaussian_kde, linregress, pearsonr, spearmanr

#Set grid colors
BG = "#fdfdfb"
GRID = "#d9d8d4"
DARK = "#333333"

#Input data path
DATA_PATH = "Data_3_Correlation_Data.xlsx"

VIRUSES = [
    ("fluA", "Influenza A", "FluA", "flua", "#3C83D9", "o"),
    ("fluB", "Influenza B", "FluB", "flub", "#EC7444", "s"),
    ("rsvA", "RSV A", "RsvA", "rsva", "#2CB483", "^"),
    ("rsvB", "RSV B", "RsvB", "rsvb", "#5748AD", "D"),
    ("sars-cov-2", "SARS-CoV-2", "SARS-COV-2", "sarscov2", "#E45655", "P"),
]

FONT_SCALE = 1.6  # 1.0 = current sizes; raise or lower to taste
def fs(size):
    return size * FONT_SCALE
    
# ----------------- Data loading ----------------- #

def load_alldata(sheet):
    df = pd.read_excel(DATA_PATH, sheet_name=sheet)
    df = df[df["mismatch_flag_after_correction"] == "none"].copy()
    x = df["copiesperml"].to_numpy(dtype=float)
    y = df["rpkmf"].to_numpy(dtype=float)
    return x, y


def load_doublepositive(sheet):
    df = pd.read_excel(DATA_PATH, sheet_name=sheet)
    df = df[df["mismatch_flag_after_correction"] == "none"]
    df = df[(df["qpcr_detected"] == "detected") & (df["sequencing_detected"] == "detected")]
    x = df["copiesperml"].to_numpy(dtype=float)
    y = df["rpkmf"].to_numpy(dtype=float)
    return x, y

RNG = np.random.default_rng(0)
RAIN_FRAC = 0.16  # band nearest the shared axis reserved for the jittered raw-point rain

def draw_marginal_kde(ax, values, orientation, color, rain=False):
    """Log-KDE half-violin. When rain=True, the violin sits above a jittered
    strip of the raw points themselves -- so the marginal shows both the
    smoothed density AND every individual data point along that axis."""
    values = np.asarray(values, dtype=float)
    logv = np.log10(values)
    kde = gaussian_kde(logv)
    grid = np.linspace(logv.min(), logv.max(), 200)
    dens = kde(grid)
    base = RAIN_FRAC if rain else 0.0
    dens = dens / dens.max() * (1 - base)
    lin_grid = 10 ** grid
    if orientation == "x":
        ax.fill_between(lin_grid, base, base + dens, color=color, alpha=0.3, linewidth=0)
        ax.plot(lin_grid, base + dens, color=color, alpha=0.95, linewidth=2)
        if rain:
            jitter = RNG.uniform(0.0, RAIN_FRAC * 0.85, size=len(values))
            ax.scatter(values, jitter, s=5, color=color, alpha=0.4, linewidth=0, zorder=2)
    else:
        ax.fill_betweenx(lin_grid, base, base + dens, color=color, alpha=0.3, linewidth=0)
        ax.plot(base + dens, lin_grid, color=color, alpha=0.95, linewidth=2)
        if rain:
            jitter = RNG.uniform(0.0, RAIN_FRAC * 0.85, size=len(values))
            ax.scatter(jitter, values, s=5, color=color, alpha=0.4, linewidth=0, zorder=2)


def style_marginal(ax):
    ax.tick_params(axis="both", which="both", bottom=False, top=False, left=False, right=False,
                    labelbottom=False, labeltop=False, labelleft=False, labelright=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_facecolor("none")


def auto_linthresh(values):
    pos = values[values > 0]
    if len(pos) == 0:
        return 1.0
    return 10 ** np.floor(np.log10(pos.min()))


def fit_line(x, y):
    log_x, log_y = np.log10(x), np.log10(y)
    res = linregress(log_x, log_y)
    log_x_line = np.linspace(log_x.min(), log_x.max(), 200)
    x_line = 10 ** log_x_line
    y_line = 10 ** (res.slope * log_x_line + res.intercept)
    return x_line, y_line, res


# ----------------- alldata figure (quad-marginal) ----------------- #

def build_quad_panel(fig, spec, label, color, marker, x, y):
    nested = GridSpecFromSubplotSpec(
        3, 3, subplot_spec=spec, width_ratios=[1, 4, 1], height_ratios=[1, 4, 1],
        hspace=0.18, wspace=0.18,
    )
    ax_main = fig.add_subplot(nested[1, 1])
    ax_top = fig.add_subplot(nested[0, 1], sharex=ax_main)
    ax_right = fig.add_subplot(nested[1, 2], sharey=ax_main)
    ax_left = fig.add_subplot(nested[1, 0], sharey=ax_main)
    ax_bottom = fig.add_subplot(nested[2, 1], sharex=ax_main)
    # unused bottom-right corner cell of the 3x3 nested grid -- used to hold
    # the legend fully outside the scatter/marginal panels.
    ax_legend = fig.add_subplot(nested[0, 2])
    ax_legend.axis("off")
    for ax in (ax_main, ax_top, ax_right, ax_left, ax_bottom):
        ax.set_facecolor(BG)

    linthresh_x = auto_linthresh(x)
    # floor at 1e-3 -- a single borderline point just under 1e-3 would
    # otherwise push auto_linthresh down a whole extra decade, adding a
    # mostly-empty decade of axis space for essentially no data.
    linthresh_y = max(auto_linthresh(y), 1e-3)

    ax_main.scatter(x, y, s=50, color=color, alpha=0.55, linewidth=0, marker=marker, zorder=3,
                     label=f"{label} (n={len(x):,})")

    x_pos, y_pos = x[x > 0], y[y > 0]
    if len(x_pos) >= 2:
        draw_marginal_kde(ax_top, x_pos, "x", color)
    if len(y_pos) >= 2:
        draw_marginal_kde(ax_right, y_pos, "y", color)

    y_given_x0 = y[(x == 0) & (y > 0)]
    if len(y_given_x0) >= 2:
        draw_marginal_kde(ax_left, y_given_x0, "y", color)
    x_given_y0 = x[(y == 0) & (x > 0)]
    if len(x_given_y0) >= 2:
        draw_marginal_kde(ax_bottom, x_given_y0, "x", color)

    ax_main.set_xscale("symlog", linthresh=linthresh_x)
    ax_main.set_yscale("symlog", linthresh=linthresh_y)
    ax_main.set_xlim(left=-linthresh_x * 0.6)
    ax_main.set_ylim(bottom=-linthresh_y * 0.6)
    ax_main.tick_params(axis="both", labelsize=fs(14))
    ax_top.set_ylim(0, 1.1)
    ax_right.set_xlim(0, 1.1)
    ax_left.set_xlim(0, 1.1)
    ax_bottom.set_ylim(0, 1.1)
    ax_left.invert_xaxis()
    ax_bottom.invert_yaxis()

    for ax in (ax_top, ax_right, ax_left, ax_bottom):
        style_marginal(ax)

    ax_main.grid(True, color=GRID, alpha=0.9, linewidth=0.9)
    ax_main.set_axisbelow(True)

    n_total = len(x)

    #Calculate correaltion coefficients (Pearsons, Spearmans)
    r, rp = pearsonr(x, y)
    rho, rhop = spearmanr(x, y)
    #stats_txt = f"n = {n_total:,}\nPearson r = {r:.3f} (p = {rp:.1e})\nSpearman ρ = {rho:.3f} (p = {rhop:.1e})"
    stats_txt = f"n = {n_total:,}\nSpearman ρ = {rho:.3f} (p = {rhop:.1e})"
    ax_main.text(0.03, 0.97, stats_txt, transform=ax_main.transAxes, va="top", ha="left",
                 fontsize=fs(16), bbox=dict(boxstyle="round", facecolor="white", edgecolor="#cccccc", alpha=1.0),
                 zorder=5)

    x_zero_share = (x == 0).mean()
    y_zero_share = (y == 0).mean()
    
    n_x0 = (x == 0).sum()
    n_y0 = (y == 0).sum()
    pct_x0_also_y0 = (y[x == 0] == 0).mean() if n_x0 else 0
    pct_y0_also_x0 = (x[y == 0] == 0).mean() if n_y0 else 0
    ax_left.text(0.03, 1.02, f"qPCR-negative, sequencing positive \n (n={n_x0:,})",
                 transform=ax_left.transAxes, ha="left", va="bottom", fontsize=fs(14), color="#555555")
    ax_bottom.text(0.0, -0.06, f"sequencing-negative, qPCR positive \n (n={n_y0:,})",
                   transform=ax_bottom.transAxes, ha="left", va="top", fontsize=fs(14), color="#555555")

    ax_bottom.text(0.5, -0.22, "copies / mL (qPCR)", transform=ax_bottom.transAxes,
                   ha="center", va="top", fontsize=fs(18), fontweight="bold", color="black")
    ax_left.text(-0.28, 0.5, "RPKMF (Sequencing)", transform=ax_left.transAxes, rotation=90,
                 ha="right", va="center", fontsize=fs(18), fontweight="bold", color="black")

    handles, labels = ax_main.get_legend_handles_labels()
    ax_legend.legend(handles, labels, loc="center", fontsize=fs(16), frameon=True, facecolor="white",
                      edgecolor="#cccccc", framealpha=1.0, title="Virus", title_fontsize=fs(18))

    return ax_main


def make_alldata_figure(label, color, marker, x, y, out_path):
    n = len(x)
    fig = plt.figure(figsize=(20, 14), facecolor=BG)
    gs = fig.add_gridspec(1, 1, left=0.10, right=0.86, top=0.80, bottom=0.07)
    build_quad_panel(fig, gs[0, 0], label, color, marker, x, y)

    fig.suptitle(f"qPCR copies/mL vs sequencing RPKMF — {label} (n={n:,})",
                 fontsize=fs(15), fontweight="bold", ha="center")

    fig.savefig(out_path, dpi=300, facecolor=fig.get_facecolor())
    fig.savefig(out_path.replace(".png", ".svg"), dpi=300, bbox_inches="tight", pad_inches=0.3)
    plt.close(fig)
    print(f"[{label}] alldata n={n:,} -> {out_path}")


# ----------------- double-positive figure (2-marginal) ----------------- #

def make_doublepositive_figure(label, color, marker, x, y, out_path):
    n = len(x)
    fig = plt.figure(figsize=(20, 14), facecolor=BG)
    gs = fig.add_gridspec(1, 1, left=0.08, right=0.97, top=0.82, bottom=0.09)
    nested = GridSpecFromSubplotSpec(
        2, 2, subplot_spec=gs[0, 0], width_ratios=[4, 1], height_ratios=[1, 4],
        hspace=0.06, wspace=0.06,
    )
    ax_top = fig.add_subplot(nested[0, 0])
    ax_main = fig.add_subplot(nested[1, 0], sharex=ax_top)
    ax_right = fig.add_subplot(nested[1, 1], sharey=ax_main)
    ax_legend = fig.add_subplot(nested[0, 1])
    ax_legend.axis("off")
    for ax in (ax_main, ax_top, ax_right):
        ax.set_facecolor(BG)

    ax_main.scatter(x, y, s=50, color=color, alpha=0.6, linewidth=0, marker=marker, zorder=3,label=f"{label} (n={n:,})")
    ax_main.tick_params(axis="both", labelsize=fs(14))
    draw_marginal_kde(ax_top, x, "x", color, rain=True)
    draw_marginal_kde(ax_right, y, "y", color, rain=True)
    ax_main.set_xscale("log")
    ax_main.set_yscale("log")

    x_line, y_line, res = fit_line(x, y)
    ax_main.plot(x_line, y_line, color=DARK, linewidth=2.5, linestyle="--", zorder=4,
                 label=f"log-log OLS fit (R²={res.rvalue**2:.3f})")

    ax_top.set_ylim(0, 1.1)
    ax_right.set_xlim(0, 1.1)
    style_marginal(ax_top)
    style_marginal(ax_right)

    r, rp = pearsonr(x, y)
    rho, rhop = spearmanr(x, y)
    #stats_txt = (f"n = {n}\nPearson r = {r:.3f} (p = {rp:.1e})\nSpearman ρ = {rho:.3f} (p = {rhop:.1e})\n"
    #             f"log-log OLS: slope = {res.slope:.3f}, R² = {res.rvalue**2:.3f}")
    stats_txt = (f"n = {n}\nSpearman ρ = {rho:.3f} (p = {rhop:.1e})\n"
                 f"log-log OLS: slope = {res.slope:.3f}, R² = {res.rvalue**2:.3f}")
    ax_main.text(0.03, 0.97, stats_txt, transform=ax_main.transAxes, va="top", ha="left",
                 fontsize=fs(16), bbox=dict(boxstyle="round", facecolor="white", edgecolor="#cccccc"))

    ax_main.set_xlabel("copies / mL (qPCR)", fontsize=fs(18), fontweight="bold")
    ax_main.set_ylabel("RPKMF (Sequencing)", fontsize=fs(18), fontweight="bold")
    ax_main.grid(True, color=GRID, alpha=0.9, linewidth=0.9)
    ax_main.set_axisbelow(True)
    handles, labels = ax_main.get_legend_handles_labels()
    ax_legend.legend(handles, labels, loc="center", fontsize=fs(16), frameon=True, facecolor="white",
                     edgecolor="#cccccc", framealpha=1.0, title="Virus", title_fontsize=fs(18))

    fig.suptitle(f"qPCR copies/mL vs sequencing RPKMF — {label}, double-positive only (n={n})",
                 fontsize=fs(18), fontweight="bold", ha="center")

    fig.savefig(out_path, dpi=300, facecolor=fig.get_facecolor())
    fig.savefig(out_path.replace(".png", ".svg"), dpi=300, bbox_inches="tight", pad_inches=0.3)
    plt.close(fig)
    print(f"[{label}] doublepositive n={n:,} -> {out_path}")


def main():
    for sheet, label, folder, slug, color, marker in VIRUSES:
        x_all, y_all = load_alldata(sheet)
        out_alldata = f"fig3B_copiesperml_vs_rpkmf_{slug}_alldata_data3.png"
        make_alldata_figure(label, color, marker, x_all, y_all, out_alldata)

        x_dp, y_dp = load_doublepositive(sheet)
        out_dp = f"fig3B_copiesperml_vs_rpkmf_{slug}_doublepositive_data3.png"
        make_doublepositive_figure(label, color, marker, x_dp, y_dp, out_dp)


if __name__ == "__main__":
    main()


[Influenza A] alldata n=3,910 -> fig3B_copiesperml_vs_rpkmf_flua_alldata_data3.png
[Influenza A] doublepositive n=264 -> fig3B_copiesperml_vs_rpkmf_flua_doublepositive_data3.png
[Influenza B] alldata n=3,897 -> fig3B_copiesperml_vs_rpkmf_flub_alldata_data3.png
[Influenza B] doublepositive n=99 -> fig3B_copiesperml_vs_rpkmf_flub_doublepositive_data3.png
[RSV A] alldata n=3,879 -> fig3B_copiesperml_vs_rpkmf_rsva_alldata_data3.png
[RSV A] doublepositive n=495 -> fig3B_copiesperml_vs_rpkmf_rsva_doublepositive_data3.png
[RSV B] alldata n=3,914 -> fig3B_copiesperml_vs_rpkmf_rsvb_alldata_data3.png
[RSV B] doublepositive n=169 -> fig3B_copiesperml_vs_rpkmf_rsvb_doublepositive_data3.png
[SARS-CoV-2] alldata n=3,910 -> fig3B_copiesperml_vs_rpkmf_sarscov2_alldata_data3.png
[SARS-CoV-2] doublepositive n=3,060 -> fig3B_copiesperml_vs_rpkmf_sarscov2_doublepositive_data3.png
